# Lab 12b : Désignation séquentielle — le contrat C4, l'orchestrateur explicite au-dessus d'ADK

Le quatrième contrat EPITA du registre (#14058) : **le runtime désigne le prochain agent selon une stratégie explicite**. La mesure initiale : ADK 2.8 embarque bien un ordonnanceur dynamique, mais aucune surface du dépôt n'exposait de stratégie — l'ordre Planner→Coder→… de vos chaînes était codé à la main, étape par étape, chez l'appelant. Ce lab travaille sur le portage livré par #14685 : un `AdkOrchestrator` accepte N spécialistes et exécute la chaîne selon un **plan déclaratif** posé avant le premier appel LLM.

LLM réel (OpenRouter) : le *contenu* de chaque étape est produit par le modèle ; l'*ordre* des étapes, lui, n'est jamais décidé par le modèle.

## 1. Configuration

In [1]:
import sys
sys.path.insert(0, '..')

import warnings

warnings.filterwarnings(
    "ignore",
    message=(
        r"\[EXPERIMENTAL\] feature "
        r"FeatureName\.JSON_SCHEMA_FOR_FUNC_DECL is enabled\."
    ),
    category=UserWarning,
    module=r"google\.adk\.models\.llm_request",
)

from config.providers import get_settings, get_provider_config, get_litellm_model

settings = get_settings()
provider = get_provider_config(settings)
print(f"Provider actif : {provider.provider.value}")
print(f"Modele : {get_litellm_model(provider)}")
print(f"Endpoint externe : {bool(provider.base_url)}")

import logging

# Runner par etape + session partagee : chaque runner previent (warning)
# pour les events de specialistes hors de son arbre -- attendu, bruit seul.
# ADK prefixe ses loggers ("google_adk." + __name__) : viser le nom reel.
logging.getLogger("google_adk.google.adk.runners").setLevel(logging.ERROR)


Provider actif : openrouter
Modele : openrouter/openai/gpt-4.1
Endpoint externe : True


## 2. La chaîne déclarée : quatre spécialistes, un plan explicite

La désignation C4 se **déclare** : la flotte d'agents d'un côté (quatre spécialistes construits par `build_agent` — l'exécuteur porte l'outil réel `dataset_profile`), le **plan** de l'autre — un tuple de noms, posé avant toute exécution. L'orchestrateur détient une session ADK unique : chaque spécialiste désigné verra tout ce que ses prédécesseurs ont produit.

In [2]:
from utils.adk_runtime import build_agent, dataset_profile
from utils.adk_orchestrator import AdkOrchestrator

planner = build_agent(
    name="planner",
    description="Planificateur de la chaîne : décompose la demande.",
    instruction=(
        "Tu es le planificateur d'une chaîne d'analyse de données. "
        "Décompose la demande en trois étapes courtes et numérotées. "
        "Ne code rien, ne calcule rien : tu planifies."
    ),
)
coder = build_agent(
    name="coder",
    description="Codeur de la chaîne : écrit la fonction demandée.",
    instruction=(
        "Tu es le codeur de la chaîne. Écris une fonction Python courte "
        "(nommée, commentée) qui réponde à l'étape en cours. Une fonction "
        "seulement, prête à exécuter."
    ),
)
executor = build_agent(
    name="executor",
    description="Exécuteur de la chaîne : mesure avec l'outil réel.",
    instruction=(
        "Tu es l'exécuteur de la chaîne. Quand un profil de dataset est "
        "en jeu, appelle OBLIGATOIREMENT l'outil dataset_profile avec les "
        "dimensions (lignes, colonnes) présentes dans la conversation, "
        "puis restitue ses chiffres."
    ),
    tools=(dataset_profile,),
)
verifier = build_agent(
    name="verifier",
    description="Vérificateur de la chaîne : clôt par un verdict.",
    instruction=(
        "Tu es le vérificateur de la chaîne. Relis ce que tes "
        "prédécesseurs ont produit et conclus en deux lignes : VERDICT "
        "(COHÉRENT ou ÉCART), puis l'argument central."
    ),
)

plan_complet = ("planner", "coder", "executor", "verifier")
chaine = AdkOrchestrator(
    [planner, coder, executor, verifier],
    plan=plan_complet,
)
print(f"Plan declaré AVANT execution : {chaine.plan}")
print("La designation est une donnée posée par l'appelant -- l'ordre "
      "n'est decidé par aucun LLM.")


Plan declaré AVANT execution : ('planner', 'coder', 'executor', 'verifier')
La designation est une donnée posée par l'appelant -- l'ordre n'est decidé par aucun LLM.


### Lecture du résultat

Le plan est imprimé **avant** l'exécution : c'est l'observable de la stratégie. Les quatre agents sont de vrais `Agent` ADK — l'orchestrateur n'assemble que des primitives publiques (un `Runner` par étape, un `InMemorySessionService` partagé), jamais une restauration du moteur SK (#14058).

## 3. La chaîne en action : chaque spécialiste prend le relais

Un seul appel, quatre étapes désignées. La première reçoit le prompt ; chacune des suivantes s'enchaîne sur l'historique de la session partagée, **sans nouveau message utilisateur** — le spécialiste désigné prend le relais sur ce qui précède. Observez `agent_hands` (la désignation *exécutée*) et `tool_calls` (l'outil réel de l'exécuteur).

In [3]:
import asyncio

async def chaine_complete():
    async with AdkOrchestrator(
        [planner, coder, executor, verifier], plan=plan_complet
    ) as orchestrateur:
        return orchestrateur, await orchestrateur.run_chain(
            "Traite ce dataset de 120 lignes et 8 colonnes : planifie "
            "l'analyse, écris la fonction de profil, exécute le profil, "
            "puis vérifie le résultat.",
            timeout_seconds=240,
        )

orchestrateur, resultat = await chaine_complete()
print(f"Désignation exécutée (mains) : {resultat.agent_hands}")
print(f"Appels d'outils : {resultat.tool_calls}")
print(f"Réponse finale (du dernier désigné, {resultat.final_agent}) :")
print(resultat.response_text[:400])


Désignation exécutée (mains) : ('planner', 'coder', 'executor', 'verifier')
Appels d'outils : ('dataset_profile',)
Réponse finale (du dernier désigné, verifier) :
VERDICT : COHÉRENT
Argument central : Le résultat du profilage correspond parfaitement à la structure attendue (120 lignes × 8 colonnes) et la fonction de profilage développée adresse bien les indicateurs planifiés, confirmant l’exécution conforme de la chaîne.


### Lecture du résultat

`agent_hands` rend la chaîne (`planner`, `coder`, `executor`, `verifier`) : l'exécution a suivi la désignation déclarée. L'exécuteur a invoqué `dataset_profile` — ses chiffres dans la réponse finale viennent de l'outil, pas d'une invention du modèle. La réponse finale est celle du **dernier désigné** (`final_agent`).

## 4. Désignation C4 vs handoff C5 : l'ordre posé avant, pas décidé pendant

Le point de distinction avec le Lab 12c : là, le transfert était **décidé par l'agent** au milieu d'un tour (`transfer_to_agent`) ; ici, l'ordre est **une donnée**. La preuve par l'expérience : la même flotte d'agents, un plan réduit — le déroulé change sans qu'aucun agent ne soit modifié.

In [4]:
async def plan_reduit():
    # MÊME flotte d'agents, AUTRE stratégie : la désignation est une
    # donnée par chaîne -- on retire coder et executor du plan sans
    # toucher aux agents eux-mêmes.
    async with AdkOrchestrator(
        [planner, coder, executor, verifier],
        plan=("planner", "verifier"),
    ) as orchestrateur:
        return orchestrateur, await orchestrateur.run_chain(
            "Prépare en une étape la vérification directe de ce dataset "
            "de 120 lignes et 8 colonnes.",
            timeout_seconds=240,
        )

orchestrateur_reduit, resultat_reduit = await plan_reduit()
print(f"Plan réduit : {orchestrateur_reduit.plan}")
print(f"Désignation exécutée : {resultat_reduit.agent_hands}")
print(f"Appels d'outils : {resultat_reduit.tool_calls or 'aucun'}")
print("Côté C4, changer l'ordre = changer UNE donnée ; côté C5 "
      "(Lab 12c), le transfert était décidé par le modèle en cours de tour.")


Plan réduit : ('planner', 'verifier')
Désignation exécutée : ('planner', 'verifier')
Appels d'outils : aucun
Côté C4, changer l'ordre = changer UNE donnée ; côté C5 (Lab 12c), le transfert était décidé par le modèle en cours de tour.


### Lecture du résultat

Deux mains seulement (`planner`, `verifier`), aucun appel d'outil : coder et executor existent toujours, ils ne sont simplement **pas désignés**. Changer l'ordre d'une chaîne C4 = changer une ligne de données ; changer le cours d'un handoff C5 = réécrire l'instruction d'un agent. Les deux contrats cohabitent sans se recouvrir.

## 5. Mémoire commune intra-chaîne, isolation inter-chaînes

La session partagée donne à la chaîne une **mémoire commune** : l'historique porte les quatre auteurs. La contre-garde C1 exige la symétrie inverse : deux chaînes sont deux sessions isolées — le message confidentiel de la première ne doit jamais atteindre l'historique de la seconde.

In [5]:
async def memoire_et_isolation():
    async with AdkOrchestrator(
        [planner, coder, executor, verifier], plan=plan_complet
    ) as chaine_a:
        await chaine_a.run_chain(
            "Analyse confidentielle : 60 lignes, 4 colonnes.",
            timeout_seconds=240,
        )
        historique_a = await chaine_a.history()
    async with AdkOrchestrator(
        [planner, coder, executor, verifier], plan=("planner",)
    ) as chaine_b:
        await chaine_b.run_chain(
            "Question banale : 10 lignes, 2 colonnes.",
            timeout_seconds=240,
        )
        historique_b = await chaine_b.history()
    return historique_a, historique_b

def textes(historique):
    return [
        "".join(p.text or "" for p in (event.content.parts or []))
        for event in historique if event.content
    ]

historique_a, historique_b = await memoire_et_isolation()
auteurs_a = [event.author for event in historique_a]
print(f"Auteurs dans la chaîne A (mémoire commune) : {auteurs_a}")
fuite = any("confidentielle" in t for t in textes(historique_b))
print(f"Le message confidentiel de A a-t-il fui dans B ? {fuite}")
print("La mémoire est commune À L'INTÉRIEUR d'une chaîne, jamais ENTRE "
      "chaînes (contre-garde C1).")


Auteurs dans la chaîne A (mémoire commune) : ['user', 'planner', 'coder', 'executor', 'executor', 'executor', 'verifier']
Le message confidentiel de A a-t-il fui dans B ? False
La mémoire est commune À L'INTÉRIEUR d'une chaîne, jamais ENTRE chaînes (contre-garde C1).


### Lecture du résultat

Côté mémoire : l'historique de la chaîne A liste bien ses auteurs dans l'ordre désigné — chacun a lu ses prédécesseurs. Côté isolation : `fuite = False` — la chaîne B n'a rien vu de la conversation de A. La mémoire est commune *à l'intérieur* d'une chaîne, jamais *entre* chaînes : c'est le contrat C1 qui reste tenu, désignation ou pas.

## 6. Exercices

### Exercice 1 — Vérificateur en tête

Déclare la même flotte avec le plan `("verifier", "planner")` et exécute-la. Que produit une vérification qui précède le plan ? Observe `agent_hands` et la cohérence du verdict final.

In [6]:
# Exercice 1 : a completer


### Exercice 2 — Le plan n'appartient pas aux agents

Construis deux `AdkOrchestrator` sur la MÊME flotte d'agents avec des plans différents, exécute les deux. Montre que le plan est une donnée par chaîne : aucun des deux déroulés ne modifie l'autre ni les agents.

In [7]:
# Exercice 2 : a completer


### Exercice 3 — Compteur de désignation

Écris `nb_etapes(resultat)` qui compte les mains distinctes d'un `AdkRunResult`. Vérifie qu'elle rend 4 pour la chaîne complète (§3) et 2 pour le plan réduit (§4).

In [8]:
# Exercice 3 : a completer


## 7. Conclusion

- **C4 est porté au-dessus d'ADK** : `AdkOrchestrator` exécute une chaîne selon un plan déclaratif — la stratégie est une **donnée** observable avant l'exécution (`orchestrator.plan`) et vérifiée après (`agent_hands`).
- **La chaîne a une mémoire commune** : chaque spécialiste désigné voit tout l'historique ; l'isolement reste scopé par session (contre-garde C1, testée au retrait).
- **C4 et C5 restent distincts** : désignation = ordre posé avant le premier appel LLM ; handoff = décision de l'agent au milieu d'un tour. L'ordonnanceur dynamique natif d'ADK reste la primitive candidate pour une désignation adaptée au contenu — une stratégie enrichie du même contrat.
- Registre vivant : #14058 — jamais une restauration SK : ADK reste le runtime.